Highest Salary for each department


In [0]:
spark.sql(""" select * from my_learning.my_raw_feed.employees""").show(truncate=False)

In [0]:
display(spark.sql("""
SELECT department_id, MAX(salary) AS second_highest_salary
FROM my_learning.my_raw_feed.employees e1
WHERE salary < (
    SELECT MAX(salary)
    FROM my_learning.my_raw_feed.employees e2
    WHERE e2.department_id = e1.department_id
)
GROUP BY department_id
"""))

In [0]:
%sql

SELECT department_id, MAX(salary) AS second_highest_salary
FROM my_learning.my_raw_feed.employees e1
WHERE (department_id,salary) not in (
    SELECT department_id,MAX(salary)
    FROM my_learning.my_raw_feed.employees e2
    group by department_id
)
GROUP BY department_id



In [0]:
%sql

select * from (select department_id,salary, row_number() over(partition by department_id order by salary desc) as rnk
from my_learning.my_raw_feed.employees) t
where rnk= 2



# find employees who are also manager

In [0]:
%sql

select manager_id from my_learning.my_raw_feed.employees where coalesce(try_cast(MANAGER_ID as int),null) is not null

In [0]:
%sql
select employee_id from my_learning.my_raw_feed.employees where EMPLOYEE_ID in (select manager_id from my_learning.my_raw_feed.employees where coalesce(try_cast(MANAGER_ID as int),null) is not null)

# Employees who earn more than thier manager

In [0]:
%sql
select employee_id ,salary from my_learning.my_raw_feed.employees 
where salary > (select salary from my_learning.my_raw_feed.employees where coalesce(try_cast(MANAGER_ID as int),null) = EMPLOYEE_ID)

In [0]:
%sql

SELECT e.EMPLOYEE_ID, e.SALARY, m.EMPLOYEE_ID AS MANAGER_ID, m.SALARY AS MANAGER_SALARY
  FROM my_learning.my_raw_feed.employees  e
  JOIN my_learning.my_raw_feed.employees  m
    ON cast(coalesce(e.MANAGER_ID,'' ) as STRING) = cast(m.EMPLOYEE_ID as STRING)
  WHERE e.SALARY > m.SALARY